In [ ]:
import re, ast
from pathlib import Path
import numpy as np
import pandas as pd

# Config
ROOT = Path(".")
RET_FILE = "portfolio_returns_test_rl.csv"
RET_COL = "ret_test_rl"

ANNUALIZATION = 252.0
RISK_FREE_DAILY = 0.0
CUM_MODE = "auto"
EPS = 1e-8

CF_KEEP = {0.0, 2.5, 5.0, 7.5, 10.0}

RUN_NAME_INCLUDE_RE = re.compile(r"^rl_runs_.*$")
RUN_NAME_EXCLUDE_RE = re.compile(r"$^")


def sharpe_ratio(daily_returns, rf_daily, annualization):
    excess = (daily_returns - rf_daily).dropna()
    if excess.empty:
        return np.nan
    sd = excess.std(ddof=1)
    if sd == 0 or not np.isfinite(sd):
        return np.nan
    return float(excess.mean() / sd * np.sqrt(annualization))

def max_drawdown(daily_returns):
    wealth = (1 + daily_returns).cumprod()
    peak = wealth.cummax()
    dd = (wealth / peak) - 1  # <= 0
    return float(dd.min()) if len(dd) else np.nan

def compute_portfolio_metrics(csv_path):
    if not csv_path.exists():
        return {"ret_error": "missing_file"}
    df = pd.read_csv(csv_path)
    if RET_COL not in df.columns:
        return {"ret_error": f"missing_col:{RET_COL}"}
    series = pd.to_numeric(df[RET_COL], errors="coerce").dropna()
    if series.empty or len(series) < 3:
        return {"ret_error": "too_short_or_empty"}

    daily_r = series

    sr = sharpe_ratio(daily_r, RISK_FREE_DAILY, ANNUALIZATION)
    mdd = max_drawdown(daily_r)  # negative
    ann_ret = float(daily_r.mean() * ANNUALIZATION)
    ann_vol = float(daily_r.std(ddof=1) * np.sqrt(ANNUALIZATION))
    calmar = float(abs(ann_ret / mdd)) if np.isfinite(mdd) and mdd != 0 else np.nan

    return {
        "n_obs_daily": int(daily_r.size),
        "sharpe": sr,
        "ann_return": ann_ret,
        "ann_vol": ann_vol,
        "max_drawdown": mdd,
        "calmar": calmar,
    }

def parse_list_cell(x):
    if x is None:
        return np.array([], dtype=float)
    try:
        if pd.isna(x):
            return np.array([], dtype=float)
    except Exception:
        pass
    s = str(x).strip()
    if s == "" or s.lower() in {"nan", "none"}:
        return np.array([], dtype=float)
    try:
        v = ast.literal_eval(s)
        arr = np.asarray(v, dtype=float)
        return np.atleast_1d(arr).ravel()
    except Exception:
        pass
    s2 = s.replace("array(", "").replace(")", "")
    s2 = s2.strip().strip("[]").replace("\n", " ").replace(",", " ")
    return np.atleast_1d(np.fromstring(s2, sep=" ")).astype(float)

def _read_csv_if_exists(path: Path):
    return pd.read_csv(path) if path.exists() else None

def parse_seed(name: str):
    m = re.search(r"seed_(\d+)", name)
    return int(m.group(1)) if m else np.nan

def parse_variant_and_sweep(name: str):
    lower = name.lower()

    if "vanilla" in lower:
        return "vanilla", None, np.nan

    # CF sweep
    if "no_cf" in lower:
        return "no_cf", "beta_cf", 0.0

    m_half = re.search(r"cf_(\d+)_half", lower)
    if m_half:
        val = float(m_half.group(1)) + 0.5
        return f"cf_{val:g}", "beta_cf", val

    m_cf = re.search(r"cf[_-]?(\d+(?:\.\d+)?)", lower)
    if m_cf:
        val = float(m_cf.group(1))
        return f"cf_{val:g}", "beta_cf", val

    if re.match(r"^rl_runs_scr_ccm_seed_\d+$", lower):
        return "full", "beta_cf", 5.0

    return "unknown", None, np.nan

def summarize_logs(run_dir: Path) -> dict:
    logs = run_dir / "logs"
    if not logs.exists():
        return {"logs_error": "missing_logs_dir"}

    out = {}

    pac_df = _read_csv_if_exists(logs / "scr_ppo_full_sim_pac_test.csv")
    if pac_df is None or pac_df.empty:
        J_real = gap = bound = np.array([], dtype=float)
    else:
        pac = pac_df.iloc[0]
        J_real = parse_list_cell(pac.get("J_real_cumulative", np.nan))
        gap   = parse_list_cell(pac.get("abs_gap_cumulative", np.nan))
        bound = parse_list_cell(pac.get("proxy_bound_cumulative", np.nan))

    out["J_real_final"] = float(J_real[-1]) if J_real.size else np.nan
    out["gap_final"]    = float(gap[-1])    if gap.size else np.nan
    out["bound_final"]  = float(bound[-1])  if bound.size else np.nan
    out["tightness_final"] = (
        out["gap_final"] / (out["bound_final"] + EPS)
        if np.isfinite(out["gap_final"]) and np.isfinite(out["bound_final"]) else np.nan
    )

    out["gap_mean"]   = float(np.nanmean(gap))   if gap.size else np.nan
    out["bound_mean"] = float(np.nanmean(bound)) if bound.size else np.nan
    out["tightness_mean"] = (
        out["gap_mean"] / (out["bound_mean"] + EPS)
        if np.isfinite(out["gap_mean"]) and np.isfinite(out["bound_mean"]) else np.nan
    )

    con_df = _read_csv_if_exists(logs / "scr_ppo_full_contraction_diag.csv")
    if con_df is not None and ("resid_l2" in con_df.columns) and len(con_df):
        resid = con_df["resid_l2"].to_numpy(float)
        out["resid_l2_auc"]   = float(np.nanmean(resid))
        out["resid_l2_final"] = float(resid[-1])
        out["resid_l2_std"]   = float(np.nanstd(resid))
    else:
        out["resid_l2_auc"] = np.nan
        out["resid_l2_final"] = np.nan
        out["resid_l2_std"] = np.nan

    to_df = _read_csv_if_exists(logs / "portfolio_turnover_test_rl.csv")
    if to_df is not None and "turnover" in to_df.columns and len(to_df):
        out["turnover_test_mean"] = float(to_df["turnover"].mean())
    else:
        out["turnover_test_mean"] = np.nan

    return out

# Universe group
EXPERIMENT_GROUP_RE = re.compile(r"^run_\d+_(high_vol|low_vol|general)$", re.IGNORECASE)

def infer_experiment_group_and_type(run_dir: Path):
    exp_group = run_dir.parent.name
    for parent in run_dir.parents:
        if EXPERIMENT_GROUP_RE.match(parent.name):
            exp_group = parent.name
            break
    eg = exp_group.lower()
    if "high_vol" in eg:
        ut = "high_vol"
    elif "low_vol" in eg:
        ut = "low_vol"
    elif "general" in eg:
        ut = "general"
    else:
        ut = "market_proxy"
    return exp_group, ut

# Aggregation (Median + IQR)
def median_iqr(x):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return (np.nan, np.nan, np.nan, np.nan)
    q1 = float(np.percentile(x, 25))
    med = float(np.percentile(x, 50))
    q3 = float(np.percentile(x, 75))
    return (med, q1, q3, float(q3 - q1))

def aggregate(df: pd.DataFrame, group_col: str, metric_cols: list) -> pd.DataFrame:
    rows = []
    for k, g in df.groupby(group_col):
        row = {group_col: k}
        for m in metric_cols:
            med, q1, q3, iqr = median_iqr(g[m].values)
            row[m + "_median"] = med
            row[m + "_q1"] = q1
            row[m + "_q3"] = q3
            row[m + "_iqr"] = iqr
        rows.append(row)

    out = pd.DataFrame(rows)
    try:
        out[group_col] = pd.to_numeric(out[group_col])
        out = out.sort_values(group_col)
    except Exception:
        out = out.sort_values(group_col)
    return out.reset_index(drop=True)

def get_ccm_reference_rows(df_all: pd.DataFrame) -> pd.DataFrame:
    # Prefer ccm runs, then vanilla, then any single row
    ref = df_all[df_all["variant"] == "full"].copy()
    if len(ref):
        return ref
    ref = df_all[df_all["variant"] == "vanilla"].copy()
    if len(ref):
        return ref
    return df_all.iloc[:1].copy()


def is_run_dir(d: Path) -> bool:
    if not d.is_dir():
        return False
    nm = d.name
    if not RUN_NAME_INCLUDE_RE.match(nm):
        return False
    if RUN_NAME_EXCLUDE_RE.search(nm):
        return False
    return (d / RET_FILE).exists() or (d / "logs").is_dir()

run_dirs = [p for p in ROOT.rglob("rl_runs_*") if is_run_dir(p)]
run_dirs = sorted(set(run_dirs))

MIN_RUNS = 2

if len(run_dirs) < MIN_RUNS:
    raise RuntimeError(
        f"Not enough folders"
    )

if not run_dirs:
    raise RuntimeError(f"No run dirs found under {ROOT.resolve()} matching rl_runs_* with {RET_FILE} or logs/")

records = []
for d in run_dirs:
    name = d.name
    seed = parse_seed(name)
    variant, sweep_type, sweep_value = parse_variant_and_sweep(name)
    experiment_group, universe_type = infer_experiment_group_and_type(d)

    rec = {
        "run_dir": str(d), "run_name": name, "seed": seed, "variant": variant, "sweep_type": sweep_type, "sweep_value": sweep_value, 
        "beta_cf": float(sweep_value) if sweep_type == "beta_cf" else np.nan, "experiment_group": experiment_group, "universe_type": universe_type,
    }
    rec.update(compute_portfolio_metrics(d / RET_FILE))
    rec.update(summarize_logs(d))
    records.append(rec)

df_runs = pd.DataFrame(records)
metric_cols = [
    "sharpe","ann_vol","max_drawdown","calmar",
    "J_real_final","gap_final","gap_mean",
    "resid_l2_auc","resid_l2_final","resid_l2_std",
    "turnover_test_mean",
]
metric_cols = [c for c in metric_cols if c in df_runs.columns]

df_for_variant = df_runs.copy()
mask_keep = (df_for_variant["sweep_type"] != "beta_cf") | (df_for_variant["beta_cf"].isin(CF_KEEP))
df_for_variant = df_for_variant[mask_keep].copy()

df_ccm_refs = get_ccm_reference_rows(df_for_variant)

df_cf5 = df_ccm_refs.copy()
df_cf5["variant"] = "cf_5"



df_for_variant = pd.concat([df_for_variant, df_cf5], ignore_index=True)
df_variant = aggregate(df_for_variant, "variant", metric_cols)

# agg_by_variant.csv
df_variant.to_csv("agg_by_variant.csv", index=False)


df_cf = df_runs.dropna(subset=["beta_cf"]).copy()
df_cf = df_cf[df_cf["beta_cf"].isin(CF_KEEP)].copy()


df_vu = df_runs.copy()
df_vu["variant_universe"] = df_vu["variant"].astype(str) + "__" + df_vu["universe_type"].astype(str)
df_vu_agg = aggregate(df_vu, "variant_universe", metric_cols)

# agg_by_variant_x_universe.csv
df_vu_agg.to_csv("agg_by_variant_x_universe.csv", index=False)


